[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/fastapi-certified/notebooks/day-01-quickstart.ipynb#scrollTo=a1b2c3d4)

---
# Day 1 · FastAPI Quickstart — Your First API in 10 Lines
**certified-journeys / fastapi-certified** · Day 1 · Foundations

> **Goal for today:** Build a working FastAPI application with multiple routes, explore automatic interactive docs, and run it end-to-end using the TestClient — all in a single notebook.


In [ ]:
%pip install -q fastapi uvicorn[standard] httpx


## Step 1 · What is FastAPI?

FastAPI is a **modern, high-performance** Python web framework for building APIs. It is built on top of **Starlette** (the ASGI toolkit) and **Pydantic** (data validation). Key characteristics:

| Feature | What it means for you |
|---|---|
| Type-hint driven | Write Python types → get validation + docs for free |
| ASGI-native | Async-first; works with uvicorn, hypercorn, gunicorn |
| OpenAPI auto-docs | `/docs` (Swagger UI) and `/redoc` generated automatically |
| Pydantic v2 core | Fast validation (Rust core), excellent error messages |
| Editor support | Full autocompletion because it uses real type hints |

FastAPI is used in production at Microsoft, Uber, Netflix, and many more. The framework's philosophy: **declare, don't configure**.


In [ ]:
# Import FastAPI and the TestClient (lets us call routes without starting a server)
from fastapi import FastAPI
from fastapi.testclient import TestClient

# Create the application instance — this is everything your API needs to start
app = FastAPI(
    title="My First FastAPI",
    description="A minimal API built during the Day 1 notebook",
    version="0.1.0",
)

# Define the root route — GET / returns a simple JSON object
@app.get("/")
def root():
    # The dict return value is automatically serialized to JSON
    return {"message": "Hello World"}

# Create a TestClient bound to our app — no uvicorn process needed
client = TestClient(app)

# Make a GET request to / and inspect the response
response = client.get("/")
print("Status code:", response.status_code)
print("JSON body:  ", response.json())


### What just happened?

- **`FastAPI()`** creates an application instance that wires together routing, validation, and OpenAPI spec generation.
- **`@app.get("/")`** registers a route handler for `GET /`. FastAPI supports all HTTP methods via decorators: `@app.post`, `@app.put`, `@app.delete`, `@app.patch`.
- **Returning a `dict`** is all it takes — FastAPI serializes it to `application/json` automatically.
- **`TestClient`** wraps the ASGI app in a `requests`-like interface so you can call routes from a notebook without starting a real server. Production uses `uvicorn app:app --reload`.


## Step 2 · Automatic Interactive Docs

One of FastAPI's headline features: **zero-config API documentation**.

When you run `uvicorn app:app --reload` locally, FastAPI exposes two interactive doc UIs:

| URL | UI | Standard |
|---|---|---|
| `/docs` | Swagger UI | OpenAPI 3.x |
| `/redoc` | ReDoc | OpenAPI 3.x |
| `/openapi.json` | Raw JSON schema | OpenAPI 3.x |

Both UIs are **generated from your route definitions** — path, method, parameters, request body schema, and response examples. You add a docstring to a route and it shows up in the docs automatically.

> In this notebook we inspect the OpenAPI schema directly to simulate what `/docs` would show.


In [ ]:
import json

# FastAPI exposes the OpenAPI schema at /openapi.json
schema_response = client.get("/openapi.json")
schema = schema_response.json()

# Print the top-level info block — same data Swagger UI shows in its header
print("OpenAPI info block:")
print(json.dumps(schema["info"], indent=2))

print("\nRegistered paths:")
for path, methods in schema["paths"].items():
    for method in methods:
        # Show the HTTP method, path, and summary for each route
        summary = methods[method].get("summary", "(no summary)")
        print(f"  {method.upper():6} {path:30} → {summary}")


### What just happened?

- FastAPI **builds the OpenAPI schema** from your code at startup — no separate YAML or JSON config required.
- **`/openapi.json`** is a live introspection endpoint. Swagger UI and ReDoc simply read and render it.
- **Docstrings** on route functions become the `summary` and `description` fields in the schema, which appear as descriptions in the interactive docs.
- Adding parameters, request bodies, or response models will **automatically extend** the schema — you never write spec files by hand.


## Step 3 · Path Parameters — Dynamic Route Segments

Path parameters embed variables directly in the URL. FastAPI extracts and validates them using Python type hints.

```
GET /items/42
       ^^  ← path parameter: item_id = 42 (int)
```

**Type conversion is automatic:** if you declare `item_id: int` and the client sends `/items/abc`, FastAPI returns a 422 Unprocessable Entity with a structured error — no `try/except` needed.


In [ ]:
# Add a second route to the same app — no need to recreate the app object
@app.get("/items/{item_id}")
def get_item(item_id: int):
    """Return metadata for a single item by its integer ID."""
    # FastAPI automatically validates that item_id can be parsed as int
    return {"item_id": item_id, "name": f"Item #{item_id}"}

# Recreate the client after adding new routes (TestClient sees the current app state)
client = TestClient(app)

# Happy path — valid integer ID
r = client.get("/items/42")
print("GET /items/42 →", r.status_code, r.json())

# Error path — non-integer triggers automatic 422 validation error
r_bad = client.get("/items/abc")
print("GET /items/abc →", r_bad.status_code)
# Show the structured error detail FastAPI returns
print("  Error detail:", r_bad.json()["detail"][0]["msg"])


### What just happened?

- **`{item_id}`** in the path template declares a dynamic segment; the matching function parameter `item_id: int` binds and validates it.
- **Type coercion is safe:** FastAPI uses Pydantic under the hood — a string `"42"` from the URL becomes Python `int` `42`, but `"abc"` causes a structured 422 response.
- The **422 error body** follows a standard schema: `{"detail": [{"loc": [...], "msg": "...", "type": "..."}]}`. Clients can parse this programmatically.
- **Docstrings on route functions** become the route summary in `/docs` — write them even for small routes.


## Step 4 · Running with uvicorn and Hot-Reload

In a real project you run FastAPI with **uvicorn**, an ASGI server:

```bash
# Start the server; assumes your file is named app.py and the FastAPI instance is called app
uvicorn app:app --reload
```

Key uvicorn flags:

| Flag | Purpose |
|---|---|
| `--reload` | Watch for file changes and restart automatically (dev only) |
| `--host 0.0.0.0` | Bind to all interfaces (needed in Docker / Colab) |
| `--port 8000` | Default port; change to avoid conflicts |
| `--workers 4` | Multiple worker processes (disable `--reload` in prod) |

> In this notebook we simulate the running server with TestClient. Below we write the minimal `app.py` content you would save to disk and run with uvicorn.


In [ ]:
# This is the full content of a minimal app.py you would run with uvicorn
MINIMAL_APP = '''
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def root():
    return {"message": "Hello World"}

@app.get("/items/{item_id}")
def get_item(item_id: int):
    return {"item_id": item_id}
'''

# Write it to disk so you can see the full file
with open("/tmp/demo_app.py", "w") as f:
    f.write(MINIMAL_APP)

print("Saved to /tmp/demo_app.py")
print("\nTo run this in a real terminal:")
print("  uvicorn demo_app:app --reload")
print("\nThen open: http://127.0.0.1:8000/docs")

# Verify the file round-trips cleanly
with open("/tmp/demo_app.py") as f:
    content = f.read()
print("\n--- /tmp/demo_app.py ---")
print(content)


### What just happened?

- A production FastAPI app is a plain Python file with an `app` instance — no magic configuration.
- **`--reload`** uses `watchfiles` under the hood; it detects any `.py` change and restarts the worker in under a second.
- In Colab or CI environments, TestClient is the right tool — **no subprocess, no port binding, no cleanup required**.
- The `uvicorn app:app` format is `module_name:variable_name` — the module is your file name (without `.py`) and the variable is your FastAPI instance.


## Step 5 · Putting It All Together — A Richer Mini-API

Let's build a slightly more complete API that exercises everything from today:
- A root route with metadata
- A collection route with an inventory
- A detail route with path parameter validation
- Docstrings wired to OpenAPI


In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

# In-memory "database" — a simple dict keyed by integer ID
ITEMS_DB = {
    1: {"name": "Laptop", "price": 999.99, "in_stock": True},
    2: {"name": "Mouse",  "price": 29.99,  "in_stock": True},
    3: {"name": "Monitor","price": 399.00, "in_stock": False},
}

mini_app = FastAPI(title="Mini Store API", version="1.0.0")

@mini_app.get("/", tags=["meta"])
def api_info():
    """Return API metadata and available endpoints."""
    return {
        "name": "Mini Store API",
        "version": "1.0.0",
        "endpoints": ["/items", "/items/{item_id}"],
    }

@mini_app.get("/items", tags=["items"])
def list_items():
    """Return all items in the store."""
    # Convert the dict values to a list for the JSON response
    return {"items": list(ITEMS_DB.values()), "count": len(ITEMS_DB)}

@mini_app.get("/items/{item_id}", tags=["items"])
def get_item(item_id: int):
    """Return a single item by its integer ID. Returns 404 if not found."""
    if item_id not in ITEMS_DB:
        # We'll handle proper HTTP errors with HTTPException in Day 2+
        return {"error": f"Item {item_id} not found"}
    return ITEMS_DB[item_id]

mini_client = TestClient(mini_app)

# Test all three routes
print("GET /       →", mini_client.get("/").json())
print()
list_resp = mini_client.get("/items")
print("GET /items  → count:", list_resp.json()["count"])
print("             items:", [i["name"] for i in list_resp.json()["items"]])
print()
print("GET /items/1 →", mini_client.get("/items/1").json())
print("GET /items/9 →", mini_client.get("/items/9").json())


### What just happened?

- **`tags=["items"]`** groups related routes in Swagger UI under a collapsible section — an easy way to organize large APIs.
- The **in-memory dict** pattern is a clean stand-in for a real database during development; just swap the dict access for a database query later.
- **Multiple app instances** work independently in a notebook — each `FastAPI()` call creates its own router, schema, and TestClient.
- We deliberately returned a plain `{"error": ...}` dict above — **Day 2 covers `HTTPException`** for proper status codes.


In [ ]:
# Challenge: Extend the mini_app with a DELETE /items/{item_id} route
# Requirements:
#   - Accept an integer item_id as a path parameter
#   - If the item exists, remove it from ITEMS_DB and return {"deleted": item_id}
#   - If the item does NOT exist, return {"error": "Item N not found"}
#   - Test both cases with mini_client

# Your solution here:
# @mini_app.delete("/items/{item_id}")
# def delete_item(item_id: int):
#     ...

# Recreate client after adding the new route:
# mini_client = TestClient(mini_app)

# Then test:
# print(mini_client.delete("/items/2").json())   # should return {"deleted": 2}
# print(mini_client.delete("/items/99").json())  # should return error


---
## Day 1 key concepts recap

| Concept | What to remember |
|---|---|
| `FastAPI()` | Creates the app instance; accepts `title`, `description`, `version` for the OpenAPI spec |
| Route decorators | `@app.get`, `@app.post`, `@app.put`, `@app.delete` — method first, path second |
| Return a dict | FastAPI serializes it to JSON automatically; no `jsonify()` or `Response()` needed |
| Path parameters | `{name}` in path template + `name: type` in function signature → automatic validation |
| Auto docs | `/docs` (Swagger UI) and `/redoc` are always available; generated from your code |
| TestClient | Use `from fastapi.testclient import TestClient` in notebooks/tests; no server process needed |
| uvicorn | `uvicorn app:app --reload` for local dev; module:variable naming convention |

> **Tip:** FastAPI auto-generates `/docs` (Swagger UI) and `/redoc` from your route definitions. You get interactive API documentation for free.

---
## What's next
**Day 2** → Path parameters, query parameters with defaults, request bodies, and combining all three in a single route.

Mark Day 1 complete in your [tracker](../index.html).
